# Linear Projection (GEMM)
*Matrix multiplication — the most compute-intensive operation in a transformer.*

**Companion kernels:** `v0_naive.py` → `sm89_v1_smem_tiled.py` → `sm89_v2_tensor_core_mma.py`


## What Is a Linear Projection?

**In plain English:** A linear layer multiplies an input matrix by a weight matrix. This is how the model transforms representations: attention projections (Q, K, V, output), feedforward up/down projections — every "thinking" step is a matrix multiply.

**Where it appears:** Every attention sublayer (4× matmuls for Q, K, V, output projection) + every FFN sublayer (2-3× matmuls). ~70% of a transformer's total FLOPs are matrix multiplies.


In [ ]:
import math, random
random.seed(42)
print('ready')

## Worked Example: $2 \times 3$ Output from $2 \times 2$ Input

$$X = \begin{bmatrix} 1.0 & 2.0 \\ 3.0 & 4.0 \end{bmatrix}, \quad W = \begin{bmatrix} 0.5 & 0.5 \\ 0.3 & -0.2 \\ -0.1 & 0.8 \end{bmatrix}$$

We compute $Y = X W^T$ (PyTorch `F.linear` convention: W rows = output directions).

$$Y_{m,n} = \sum_{k=0}^{K-1} X_{m,k} \cdot W_{n,k}$$

| $Y[m,n]$ | Dot product | Result |
|-----------|-------------|--------|
| $Y[0,0]$ | $1.0×0.5 + 2.0×0.5$ | **1.5** |
| $Y[0,1]$ | $1.0×0.3 + 2.0×(-0.2)$ | **-0.1** |
| $Y[0,2]$ | $1.0×(-0.1) + 2.0×0.8$ | **1.5** |
| $Y[1,0]$ | $3.0×0.5 + 4.0×0.5$ | **3.5** |
| $Y[1,1]$ | $3.0×0.3 + 4.0×(-0.2)$ | **0.1** |
| $Y[1,2]$ | $3.0×(-0.1) + 4.0×0.8$ | **2.9** |


In [ ]:
X = [[1.0, 2.0], [3.0, 4.0]]
W = [[0.5, 0.5], [0.3, -0.2], [-0.1, 0.8]]
M, K, N = 2, 2, 3

Y = [[sum(X[m][k] * W[n][k] for k in range(K)) for n in range(N)] for m in range(M)]

print("Y = X @ W.T:")
for m, row in enumerate(Y):
    print(f"  row {m}: {[round(v,4) for v in row]}")

# Spot-check
assert abs(Y[0][0] - 1.5) < 1e-9 and abs(Y[1][2] - 2.9) < 1e-9
print("\n✓ Spot-checks pass")


## The Formula

$$\boxed{Y_{m,n} = \sum_{k=0}^{K-1} X_{m,k} \cdot W_{n,k}}$$

Equivalently: $Y = X W^T$ where $X \in \mathbb{R}^{M \times K}$, $W \in \mathbb{R}^{N \times K}$, $Y \in \mathbb{R}^{M \times N}$.

| Symbol | Meaning | In our example |
|--------|---------|----------------|
| $M$ | Batch size (number of input vectors) | 2 |
| $N$ | Output dimension (number of neurons) | 3 |
| $K$ | Input dimension | 2 |
| $X_{m,k}$ | Element $k$ of input vector $m$ | $X[0,0]=1.0$ |
| $W_{n,k}$ | Weight for output $n$ from input $k$ | $W[0,0]=0.5$ |

### 🗣️ Say It Out Loud
> *"Y at row m, column n equals the sum over k of X at m-comma-k times W at n-comma-k. This is the dot product of input row m with weight row n."*

**Tutor's gloss:** "Think of each row of W as a 'detector'. $W[0,:]$ detects the pattern $[0.5, 0.5]$ — it responds to the average of the two input features. The dot product with $X[0,:]=[1,2]$ gives $0.5×1 + 0.5×2 = 1.5$ — the detector fired."


## Arithmetic Intensity and the Cache Problem

**Arithmetic intensity** = FLOPs / bytes transferred.

For one $M \times N \times K$ matrix multiply:
$$I = \frac{2MNK}{4(MK + NK + MN)} \approx \frac{MNK}{2(MK + NK + MN)}$$

For large square matrices ($M = N = K = n$):
$$I \approx \frac{n^3}{6n^2} = \frac{n}{6} \text{ FLOP/byte}$$

| Matrix size $n$ | Intensity | Status on RTX 4080 (ridge ≈ 230) |
|----------------|-----------|----------------------------------|
| 64 | 10.7 FLOP/byte | Memory-bound (100× below ridge) |
| 1024 | 170 FLOP/byte | Approaching ridge |
| 4096 | 682 FLOP/byte | Strongly compute-bound |

**The cache problem:** In `v0_naive`, each thread fetches a different row of $W$ from HBM — zero reuse. Row $X[m,:]$ is re-fetched once per output column $n$. With $N=4096$ output columns, that's 4096 redundant HBM reads of the same row.

**The tiling fix:** All $B_M \times B_N$ threads in a CTA load the *same* $B_M \times B_K$ tile of X and $B_N \times B_K$ tile of W into shared memory once, then compute from SMEM (30–100× faster than HBM).


## Optimization Ladder

| Version | Compute path | Reuse | Approx. TFLOPS |
|---------|-------------|-------|----------------|
| `v0_naive` | CUDA scalar FMA | None | < 0.1 |
| `sm89_v1_smem_tiled` | CUDA scalar FMA | $B_M$ and $B_N$ | 1–5 |
| `sm89_v2_tensor_core_mma` | m16n8k16 TC instruction | $B_M \times B_N$ | 50–165 |

**Tensor Core instruction (SM89):**
`D[16,8] += A[16,16] × B[16,8]` in fp16 — 4096 FLOPs per warp per cycle.
In CuTeDSL: `cute.gemm(tiled_mma, rA, rB, rC)` emits this PTX.
